In [49]:
from orion.utils.misc_utils import *
from orion.utils.o3d_utils import *
import numpy as np
import torch
import json
import h5py
import cv2
import sys
from PIL import Image
import PIL
dust3r_path = './third_party/dust3r'
sys.path.append(dust3r_path)
from dust3r.utils.geometry import geotrf
from tqdm import tqdm

In [50]:
def draw_scene_3d(
                      pcd_colors,
                      pcd_points,
                      draw_trajs=False,
                      draw_keypoints=False,
                      additional_points=None,
                      downsample=True,
                      depth_trunc=3.0,
                      no_background=False,
                      overlay_annotation=False,
                      additional_point_draw_lines=False,
                      use_white_bg=False,
                      skip_trajs=[],
                      skip_keypoints=[],
                      offline=False,
):

        marker_size = 3
        if offline:
            fig = plotly_draw_3d_pcd(pcd_points, pcd_colors, addition_points=additional_points, marker_size=marker_size, no_background=no_background, additional_point_draw_lines=additional_point_draw_lines, offline=offline)
            return fig
        else:
            plotly_draw_3d_pcd(pcd_points, pcd_colors, addition_points=additional_points, marker_size=marker_size, no_background=no_background, additional_point_draw_lines=additional_point_draw_lines)
            return pcd_points, pcd_colors

In [51]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/iphone_front_boat_demo"
demo_name = annotation_folder.split("/")[-1]
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)

In [52]:
with h5py.File(config_dict["original_file"], "r") as demo_file:
    #print(demo_file["data/human_demo/obs/dust3r_depth"].shape)
    scene_points = demo_file["data/human_demo/obs/scene_points"][()]
    scene_col = demo_file["data/human_demo/obs/scene_col"][()]
    dust3r_idx_list = demo_file["data/human_demo/obs/dust3r_idx_list"][()]
    dust3r_depth = demo_file["data/human_demo/obs/dust3r_depth"][()] * 1000
    dust3r_rgb = demo_file["data/human_demo/obs/dust3r_rgb"][()]
    dust3r_intrinsics = demo_file["data/human_demo/obs/dust3r_intrinsics"][()]
    dust3r_extrinsics = demo_file["data/human_demo/obs/dust3r_extrinsics"][()]

    original_rgb = demo_file["data/human_demo/obs/agentview_rgb"][()]
masks_orig = np.load(os.path.join(annotation_folder, 'masks.npz'))
masks_orig = masks_orig['arr_0']

In [ ]:
dust3r_rgb.shape

In [ ]:
print(dust3r_intrinsics[0])
print(scene_points.shape)


In [55]:
cotracker_dict = torch.load(os.path.join(annotation_folder, "tracked_points.pt"), map_location=torch.device('cuda:3'))
trajs_2d = cotracker_dict['pred_tracks'].cpu().detach().numpy()[0]
trajs_2d = trajs_2d[dust3r_idx_list]


In [56]:
dust3r_height, dust3r_width = dust3r_rgb.shape[1], dust3r_rgb.shape[2]
original_height, original_width = original_rgb.shape[1], original_rgb.shape[2]

scale_x = dust3r_width / original_width
scale_y = dust3r_height / original_height

trajs_2d[:, :, 0] = trajs_2d[:, :, 0] * scale_x
trajs_2d[:, :, 1] = trajs_2d[:, :, 1] * scale_y

# print(dust3r_height, dust3r_width)
# print(original_height, original_width)

In [ ]:
tst_img = (dust3r_rgb[0] * 255).astype(np.uint8)
tst_traj = trajs_2d[0]
for i in range(tst_traj.shape[0]):
    cv2.circle(tst_img, (int(tst_traj[i, 0]), int(tst_traj[i, 1])), 3, (0, 255, 0), -1)
display(Image.fromarray(tst_img))

In [ ]:
scene_pcd_wrapped = O3DPointCloud()
scene_pcd_wrapped.pcd = create_o3d_from_points_and_color(scene_points, scene_col)
scene_pcd_wrapped.max_points = 10000
scene_pcd_wrapped.preprocess()
scene = plotly_draw_3d_pcd(scene_pcd_wrapped.get_points(), scene_pcd_wrapped.get_colors(), marker_size=3, no_background=False, offline=True)
display(scene)

In [ ]:
#filter out trajectories that have coordinates that are out of bounds
max_x, max_y = dust3r_rgb[0].shape[1] - 1, dust3r_rgb[0].shape[0] - 1

trajs_2d_filtered = []
valid_traj_indices = []
for traj in trajs_2d:
    valid_indices = np.all((traj >= 0) & (traj <= [max_x, max_y]), axis=1)
    trajs_2d_filtered.append(traj[valid_indices])
    valid_traj_indices.append(np.argwhere(valid_indices == True))

print(len(valid_traj_indices), valid_traj_indices[0][0:5])

In [ ]:
traj_2d_pts = [[] for _ in range(trajs_2d.shape[1])]
for i in range(len(trajs_2d_filtered)):
    curr_frame_pts = trajs_2d_filtered[i]
    for idx, point in enumerate(curr_frame_pts):
        traj_2d_pts[valid_traj_indices[i][idx].item()].append(point)

test = np.array(traj_2d_pts[0], dtype=np.int32)
img = (dust3r_rgb[0] * 255).astype(np.uint8)
for i in range(trajs_2d.shape[0]):
    img = cv2.circle(img, (test[i, 0], test[i, 1]), 3, (0, 0, 255), -1)
display(Image.fromarray(img))

In [ ]:
traj_3d_pts = [[] for _ in range(trajs_2d.shape[1])]
for i in range(len(trajs_2d_filtered)):
    untransformed_trajs = create_point_clouds_from_keypoints(trajs_2d_filtered[i], dust3r_depth[i], dust3r_intrinsics[i])
    #untransformed_trajs *= 1000
    transformed_trajs = geotrf(dust3r_extrinsics[i], untransformed_trajs)
    assert untransformed_trajs.shape == transformed_trajs.shape
    for idx, point in enumerate(transformed_trajs):
        traj_3d_pts[valid_traj_indices[i][idx].item()].append(point)
    #traj_3d_pts.append(transformed_trajs)

traj_3d_pts = [np.array(traj) for traj in traj_3d_pts]
print(len(traj_3d_pts))
print([traj_3d_pts[x].shape for x in range(5)])


In [62]:
concatenated_traj_3d_pts = []
for i in range(len(traj_3d_pts)):
    if len(traj_3d_pts[i]) != 0:
        concatenated_traj_3d_pts.append(traj_3d_pts[i])
traj_3d_no_empty_pts = concatenated_traj_3d_pts
concatenated_traj_3d_pts = np.concatenate(concatenated_traj_3d_pts, axis=0)

In [ ]:
scene_pcd = o3d.geometry.PointCloud()
scene_pcd.points = o3d.utility.Vector3dVector(scene_points)
scene_pcd.colors = o3d.utility.Vector3dVector(scene_col)

frame_idx = 6
pcd_wrapped = O3DPointCloud()
pcd_wrapped.create_from_rgbd((dust3r_rgb[frame_idx] * (255 / dust3r_rgb[frame_idx].max())).astype(np.uint8), dust3r_depth[frame_idx], dust3r_intrinsics[0], depth_trunc=.3)
#pcd_wrapped.pcd = scene_pcd
pcd_wrapped.max_points = 40000


print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

plane_estimation_kwargs={
        "ransac_n": 3,
        "num_iterations": 1000,
        "distance_threshold": 0.01
    }

plane_estimation_result = pcd_wrapped.plane_estimation(**plane_estimation_kwargs)
T_xy_plane_align = estimate_rotation(plane_estimation_result["plane_model"], z_up=False)
pcd_wrapped.pcd.transform(T_xy_plane_align)

transformed_traj = []
for i in range(len(traj_3d_no_empty_pts)):
        segment = traj_3d_no_empty_pts[i]
        #segment = simple_filter_outliers(segment)
        segment = transform_points(T_xy_plane_align, segment)
        transformed_traj.append(segment)

#fig = draw_scene_3d(np.array(pcd_wrapped.pcd.colors), np.array(pcd_wrapped.pcd.points), offline=True, downsample=True)
fig = draw_scene_3d(np.array(pcd_wrapped.pcd.colors), np.array(pcd_wrapped.pcd.points), draw_trajs=False, additional_points=[transformed_traj[41]], additional_point_draw_lines=True, offline=True, downsample=True)
display(fig)